# 03 - Data Preprocessing

## Goal
In this notebook, we prepare the medical appointment no-show dataset for machine learning.  
We will remove leakage-prone variables, engineer useful features from date columns, handle missing values, and define the feature set that will be used in the modeling stage.

In [1]:
import pandas as pd

df = pd.read_csv("../data/medical-appointments-no-show-en.csv")
df.head()

,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,date_of_birth,entry_service_date,city,...,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity
0,physiotherapy,13:20,M,09/09/2021,yes,surto,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
1,psychotherapy,13:20,M,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
2,speech therapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
3,physiotherapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
4,physiotherapy,14:00,M,09/09/2021,no,NaN,motor,10/10/1954,5/2/2020,B. CAMBORIU,...,1,1,20.75,0.01,23.7,0.2,1,1,no_rain,mild


## Initial Feature Decisions

Before preparing the data for modeling, we identify variables that should not be used directly:

- `no_show_reason` will be excluded because it becomes available only after a no-show occurs. Using it would cause **data leakage**.
- `date_of_birth` will not be used directly, since `age` already captures the most relevant information in a more model-friendly form.

Additional decisions about date-related columns and high-missingness features will be made during preprocessing.

In [2]:
df = df.drop(columns=["no_show_reason", "date_of_birth"])

df.shape

(49593, 24)

In [3]:
df.columns.tolist()

['specialty',
 'appointment_time',
 'gender',
 'appointment_date',
 'no_show',
 'disability',
 'entry_service_date',
 'city',
 'icd',
 'appointment_month',
 'appointment_year',
 'appointment_shift',
 'age',
 'under_12_years_old',
 'over_60_years_old',
 'patient_needs_companion',
 'average_temp_day',
 'average_rain_day',
 'max_temp_day',
 'max_rain_day',
 'rainy_day_before',
 'storm_day_before',
 'rain_intensity',
 'heat_intensity']

In [5]:
df["appointment_date"] = pd.to_datetime(
    df["appointment_date"],
    dayfirst=True
)

In [6]:
df["appointment_day_of_week"] = df["appointment_date"].dt.day_name()

In [7]:
df[["appointment_date", "appointment_day_of_week"]].head()

,appointment_date,appointment_day_of_week
0,2021-09-09,Thursday
1,2021-09-09,Thursday
2,2021-09-09,Thursday
3,2021-09-09,Thursday
4,2021-09-09,Thursday


In [8]:
df["appointment_time"].head(15)

0     13:20
1     13:20
2     13:20
3     13:20
4     14:00
5     14:00
6     14:00
7     14:00
8     14:00
9     14:00
10    14:00
11    14:40
12    14:40
13    14:40
14    14:40
Name: appointment_time, dtype: str

In [9]:
df["appointment_time"].unique()[:20]

<ArrowStringArray>
['13:20', '14:00', '14:40', '15:20', '16:20', '17:00', '17:40', '07:20',
 '08:00', '08:40', '10:20', '11:00', '11:40', '09:20', '13:00', '13:30',
 '16:40', '16:00', '15:00', '15:40']
Length: 20, dtype: str

In [10]:
df["appointment_hour"] = (
    df["appointment_time"]
    .str.split(":")
    .str[0]
    .astype(int)
)

df[["appointment_time", "appointment_hour"]].head()

,appointment_time,appointment_hour
0,13:20,13
1,13:20,13
2,13:20,13
3,13:20,13
4,14:00,14


In [11]:
df["entry_service_date"] = pd.to_datetime(
    df["entry_service_date"],
    dayfirst=True,
    errors="coerce"
)

In [12]:
df[["appointment_date", "entry_service_date"]].head(10)

,appointment_date,entry_service_date
0,2021-09-09,NaT
1,2021-09-09,NaT
2,2021-09-09,NaT
3,2021-09-09,NaT
4,2021-09-09,2020-02-05
5,2021-09-09,2019-11-26
6,2021-09-09,2019-10-01
7,2021-09-09,NaT
8,2021-09-09,2019-11-20
9,2021-09-09,NaT


In [13]:
df = df.drop(columns=[
    "entry_service_date",
    "appointment_date",
    "appointment_time"
])

df.columns.tolist()

['specialty',
 'gender',
 'no_show',
 'disability',
 'city',
 'icd',
 'appointment_month',
 'appointment_year',
 'appointment_shift',
 'age',
 'under_12_years_old',
 'over_60_years_old',
 'patient_needs_companion',
 'average_temp_day',
 'average_rain_day',
 'max_temp_day',
 'max_rain_day',
 'rainy_day_before',
 'storm_day_before',
 'rain_intensity',
 'heat_intensity',
 'appointment_day_of_week',
 'appointment_hour']

In [14]:
df = df.drop(columns=["icd"])

df.columns.tolist()

['specialty',
 'gender',
 'no_show',
 'disability',
 'city',
 'appointment_month',
 'appointment_year',
 'appointment_shift',
 'age',
 'under_12_years_old',
 'over_60_years_old',
 'patient_needs_companion',
 'average_temp_day',
 'average_rain_day',
 'max_temp_day',
 'max_rain_day',
 'rainy_day_before',
 'storm_day_before',
 'rain_intensity',
 'heat_intensity',
 'appointment_day_of_week',
 'appointment_hour']

In [15]:
df["no_show"] = df["no_show"].map({
    "no": 0,
    "yes": 1
})

df["no_show"].value_counts()

no_show
0    44761
1     4832
Name: count, dtype: int64

In [16]:
df.isna().sum().sort_values(ascending=False)

age                        10350
specialty                   7454
city                        5181
disability                  5137
average_rain_day            1016
max_rain_day                1016
max_temp_day                1016
average_temp_day            1016
appointment_day_of_week        0
heat_intensity                 0
rain_intensity                 0
storm_day_before               0
rainy_day_before               0
patient_needs_companion        0
gender                         0
over_60_years_old              0
under_12_years_old             0
appointment_shift              0
appointment_year               0
appointment_month              0
no_show                        0
appointment_hour               0
dtype: int64

In [17]:
categorical_missing_cols = ["specialty", "city", "disability"]

for col in categorical_missing_cols:
    df[col] = df[col].replace("", pd.NA)
    df[col] = df[col].fillna("Missing")

In [18]:
df[categorical_missing_cols].isna().sum()

specialty     0
city          0
disability    0
dtype: int64

In [19]:
df["age_missing"] = df["age"].isna()

age_missing_summary = (
    df.groupby("age_missing")["no_show"]
    .agg(
        total_appointments="count",
        no_show_count="sum",
        no_show_rate=lambda x: x.mean() * 100
    )
)

age_missing_summary["no_show_rate"] = age_missing_summary["no_show_rate"].round(2)

age_missing_summary

,total_appointments,no_show_count,no_show_rate
age_missing,,,
False,39243,3494,8.90
True,10350,1338,12.93


In [20]:
weather_cols = [
    "average_rain_day",
    "max_rain_day",
    "max_temp_day",
    "average_temp_day"
]

df["weather_missing"] = df[weather_cols].isna().any(axis=1)

weather_missing_summary = (
    df.groupby("weather_missing")["no_show"]
    .agg(
        total_appointments="count",
        no_show_count="sum",
        no_show_rate=lambda x: x.mean() * 100
    )
)

weather_missing_summary["no_show_rate"] = weather_missing_summary["no_show_rate"].round(2)

weather_missing_summary

,total_appointments,no_show_count,no_show_rate
weather_missing,,,
False,48577,4666,9.61
True,1016,166,16.34


### Missingness as a Predictive Signal

Missing values appear to carry meaningful information in this dataset.

- Missing `age` is associated with a higher no-show rate (**12.93%**) compared to recorded age (**8.90%**).
- Missing weather measurements are associated with a notably higher no-show rate (**16.34%**) compared to records with available weather data (**9.61%**).

For this reason, we retain `age_missing` and `weather_missing` as explicit features for modeling.

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49593 entries, 0 to 49592
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   specialty                49593 non-null  str    
 1   gender                   49593 non-null  str    
 2   no_show                  49593 non-null  int64  
 3   disability               49593 non-null  str    
 4   city                     49593 non-null  str    
 5   appointment_month        49593 non-null  str    
 6   appointment_year         49593 non-null  int64  
 7   appointment_shift        49593 non-null  str    
 8   age                      39243 non-null  float64
 9   under_12_years_old       49593 non-null  int64  
 10  over_60_years_old        49593 non-null  int64  
 11  patient_needs_companion  49593 non-null  int64  
 12  average_temp_day         48577 non-null  float64
 13  average_rain_day         48577 non-null  float64
 14  max_temp_day             48577 no

In [22]:
target = "no_show"

categorical_features = [
    "specialty",
    "gender",
    "disability",
    "city",
    "appointment_month",
    "appointment_shift",
    "rain_intensity",
    "heat_intensity",
    "appointment_day_of_week"
]

numeric_features = [
    "appointment_year",
    "age",
    "average_temp_day",
    "average_rain_day",
    "max_temp_day",
    "max_rain_day",
    "appointment_hour"
]

binary_features = [
    "under_12_years_old",
    "over_60_years_old",
    "patient_needs_companion",
    "rainy_day_before",
    "storm_day_before",
    "age_missing",
    "weather_missing"
]

In [23]:
len(categorical_features), len(numeric_features), len(binary_features)

(9, 7, 7)

In [24]:
df.to_csv("../data/medical_no_show_preprocessed.csv", index=False)

## Preprocessing Summary

In this notebook, the dataset was prepared for machine learning by:

- Removing leakage-prone and unnecessary columns:
  - `no_show_reason`
  - `date_of_birth`
  - `entry_service_date`
  - raw `appointment_date`
  - raw `appointment_time`
  - `icd`

- Engineering useful time-related features:
  - `appointment_day_of_week`
  - `appointment_hour`

- Encoding the target variable:
  - `no_show = 1` for missed appointments
  - `no_show = 0` for attended appointments

- Handling categorical missing values by preserving them as a separate `"Missing"` category:
  - `specialty`
  - `city`
  - `disability`

- Creating explicit missingness indicators:
  - `age_missing`
  - `weather_missing`

- Keeping numeric missing values unchanged for now, so they can be handled properly inside the machine learning pipeline without data leakage.

The resulting dataset is ready to be used in the modeling stage.